In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Récupère la session Spark active
spark = SparkSession.builder.getOrCreate()

# Récupère dbutils de façon compatible (Databricks SDK / PySpark)
try:
    from databricks.sdk.runtime import dbutils
except ImportError:
    try:
        from pyspark.dbutils import DBUtils
        dbutils = DBUtils(spark)
    except ImportError:
        pass  # En environnement Databricks natif, dbutils est déjà injecté

spark.sql("USE CATALOG main")
spark.sql("USE SCHEMA bronze")

df_bronze = spark.table("transactions_bronze")

df_silver = (
    df_bronze
    # Nettoyage des valeurs nulles
    .na.drop(subset=["Amount", "Class"])

    # Typage
    .withColumn("Amount", F.col("amount").cast("double"))
    .withColumn("is_fraud", F.col("Class").cast("int"))

    # Suppression des doublons
    .dropDuplicates()

    # Suppression de la colonne Class
    .drop("Class")

    # Renommer les colonnes
    .withColumnRenamed("Amount", "amount")
    .withColumnRenamed("Time", "time")
)

# Ecriture Silver dans Unit Catalog
spark.sql("USE CATALOG main")
spark.sql("USE SCHEMA silver")

df_silver.write.format("delta").mode("overwrite").saveAsTable("transactions_silver")  # noqa: E501